# 03 — Training AryColBring: Hyperparameters & Baseline Comparison

**Objective**: sweep a small hyperparameter grid and compare AryColBring against a trivial random-recommendation baseline, using Precision@K.

**Audience**: users who already have a prepared `interactions` matrix (see `02_Data_Preparation.ipynb`) and want to pick reasonable hyperparameters before a full training run.

> This notebook is about the *usage/tuning* layer. For a section-by-section walkthrough of what `AryColBringModelTrainer.fit()` does internally, see `AryColBring_Training_Pipeline.ipynb`. As in that notebook, the runnable cells below use a lightweight synthetic SGD trainer that reproduces the real scoring formula (dot product + biases) and hyperparameter surface (`no_components`, `learning_rate`, `epochs`), since the real trainer needs the compiled `CLproximity` Cython extension.

## 1. Setup + synthetic ground truth

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np

rng = np.random.default_rng(0)
N_USERS, N_ITEMS, TRUE_DIM = 120, 40, 6

# "True" latent structure the training data is generated from.
true_user = rng.normal(size=(N_USERS, TRUE_DIM))
true_item = rng.normal(size=(N_ITEMS, TRUE_DIM))
true_scores = true_user @ true_item.T

# Turn the top few true-affinity items per user into observed "interactions".
interactions = []
for u in range(N_USERS):
    liked = np.argsort(true_scores[u])[::-1][:8]
    interactions += [(u, int(i)) for i in liked]
print(f"{len(interactions)} synthetic interactions across {N_USERS} users, {N_ITEMS} items.")

## 2. A minimal trainable model (same hyperparameter surface as `AryColBringModelTrainer`)

Real hyperparameters we're sweeping (`src/models/arycolbring/trainer.py` / `inout/scaffold.py`): `no_components`, `learning_rate`, and epochs -- everything below maps 1:1 onto those names.

In [ ]:
class MiniACB:
    """Minimal SGD matrix-factorization trainer -- same scoring formula
    (dot product + biases) as AryColBringPredictor.predict(), same
    core hyperparameters as AryColBringModelTrainer."""

    def __init__(self, n_users: int, n_items: int, no_components: int = 8,
                 learning_rate: float = 0.05, random_state: int = 0):
        r = np.random.default_rng(random_state)
        self.no_components = no_components
        self.learning_rate = learning_rate
        self.user_embeddings = r.normal(scale=0.1, size=(n_users, no_components))
        self.item_embeddings = r.normal(scale=0.1, size=(n_items, no_components))
        self.user_biases = np.zeros(n_users)
        self.item_biases = np.zeros(n_items)

    def fit(self, pairs: List[Tuple[int, int]], epochs: int = 20,
            n_negatives: int = 4, seed: int = 0) -> List[float]:
        """BPR-style pairwise loss: pull observed (u,i) score above a
        random negative item's score."""
        r = np.random.default_rng(seed)
        n_items = self.item_embeddings.shape[0]
        losses = []
        for _ in range(epochs):
            epoch_loss = 0.0
            for u, pos in pairs:
                for _ in range(n_negatives):
                    neg = int(r.integers(n_items))
                    pos_score = self.user_embeddings[u] @ self.item_embeddings[pos] \
                                + self.item_biases[pos]
                    neg_score = self.user_embeddings[u] @ self.item_embeddings[neg] \
                                + self.item_biases[neg]
                    diff = pos_score - neg_score
                    grad = 1.0 / (1.0 + np.exp(np.clip(diff, -30, 30)))  # sigmoid(-diff)
                    epoch_loss += -np.log(1e-9 + 1.0 / (1.0 + np.exp(np.clip(-diff, -30, 30))))

                    du = grad * (self.item_embeddings[pos] - self.item_embeddings[neg])
                    self.user_embeddings[u] += self.learning_rate * du
                    self.item_embeddings[pos] += self.learning_rate * grad * self.user_embeddings[u]
                    self.item_embeddings[neg] -= self.learning_rate * grad * self.user_embeddings[u]
                    self.item_biases[pos] += self.learning_rate * grad
                    self.item_biases[neg] -= self.learning_rate * grad
            losses.append(epoch_loss / (len(pairs) * n_negatives))
        return losses

    def recommend(self, user_id: int, n_items: int = 10) -> List[int]:
        scores = self.item_embeddings @ self.user_embeddings[user_id] + self.item_biases
        return list(np.argsort(scores)[::-1][:n_items])


print("MiniACB defined.")

## 3. Precision@K -- against a random baseline

`Precision@K` = (# recommended items in the user's true top-N) / K. We compare `MiniACB` (trained) against a random-item baseline, exactly the comparison Task 4 asked for.

In [ ]:
def precision_at_k(recommend_fn, k: int = 5, n_eval_users: int = 40) -> float:
    hits = 0
    for u in range(n_eval_users):
        true_top = set(np.argsort(true_scores[u])[::-1][:k].tolist())
        recommended = set(recommend_fn(u)[:k])
        hits += len(true_top & recommended)
    return hits / (n_eval_users * k)


def random_baseline(user_id: int, n_items: int = 10) -> List[int]:
    return list(rng.choice(N_ITEMS, size=n_items, replace=False))


baseline_p5 = precision_at_k(random_baseline, k=5)
print(f"Random baseline  Precision@5 = {baseline_p5:.3f}")

## 4. Hyperparameter sweep

In [ ]:
grid = [
    {"no_components": 4,  "learning_rate": 0.05},
    {"no_components": 8,  "learning_rate": 0.05},
    {"no_components": 16, "learning_rate": 0.05},
    {"no_components": 8,  "learning_rate": 0.15},
]

results = []
for params in grid:
    model = MiniACB(N_USERS, N_ITEMS, **params, random_state=1)
    model.fit(interactions, epochs=15)
    p5 = precision_at_k(model.recommend, k=5)
    results.append({**params, "precision_at_5": p5})
    print(f"no_components={params['no_components']:>2}  "
          f"learning_rate={params['learning_rate']:.2f}  "
          f"Precision@5={p5:.3f}")

best = max(results, key=lambda r: r["precision_at_5"])
print(f"\nBest config: {best}")
print(f"Improvement over random baseline: "
      f"{best['precision_at_5'] - baseline_p5:+.3f}")

## Summary

- Compared a trained recommender against a random baseline using Precision@K -- the trained model should clearly beat random on this synthetic (but structured) data.
- Swept `no_components` and `learning_rate`, the same two names `AryColBringModelTrainer` exposes.
- In a real environment, replace `MiniACB` with `AryColBringModelTrainer` and `precision_at_k`'s ground truth with `trainer.evaluate(test_interactions)` (see `src/models/arycolbring/eval/`), which reports `precision_at_k`/`recall_at_k`/`auc`/`ndcg_at_k` for you.

**Next**: `04_Interactive_Dashboard.ipynb` to visualize a trained model's predictions.